In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [1]:
import pandas as pd
import numpy as np
import csv
import pickle
import copy
import re
import random
import matplotlib.pyplot as plt
import itertools
import json
import openai
import time


In [2]:
!rm -rf LLM4BEAR
!git clone --depth 1 --filter=blob:none --sparse https://github.com/anon5159753/LLM4BEAR.git
!cd LLM4BEAR && git sparse-checkout set "BundleRec Data"

Cloning into 'LLM4BEAR'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 21 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 7.92 KiB | 3.96 MiB/s, done.
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 67 bytes | 33.00 KiB/s, done.
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 82 (delta 16), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 45.90 MiB | 9.71 MiB/s, done.
Resolving deltas: 100% (16/16), done.
Updating files: 100% (83/83), done.


In [16]:
clothing_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/clothing/bundle_list_items.pkl")

electronic_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/electronic/bundle_list_items.pkl")

food_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/food/bundle_list_items.pkl")

bundle_items_list = [clothing_bundles_items, electronic_bundles_items, food_bundles_items]

clothing_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/bundle_intent.csv")

electronics_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/bundle_intent.csv")

food_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/bundle_intent.csv")

intents = [clothing_intent, electronics_intent, food_intent]

clothing_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/merged_metadata.csv")

electronic_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/merged_metadata.csv")

food_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/merged_metadata.csv")

metadata = [clothing_metadata, electronic_metadata, food_metadata]

clothing_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_session.csv")

clothing_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_item.csv")

clothing_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_bundle.csv")

clothing_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/session_bundle.csv")

clothing_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/session_item.csv")

clothing_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/item_titles.csv")

electronic_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_session.csv")

electronic_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_item.csv")

electronic_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_bundle.csv")

electronic_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/session_bundle.csv")

electronic_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/session_item.csv")

electronic_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/item_titles.csv")

food_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_session.csv")

food_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_item.csv")

food_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_bundle.csv")

food_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/session_bundle.csv")

food_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/session_item.csv")

food_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/item_titles.csv")

user_session = [clothing_user_session, electronic_user_session, food_user_session]

user_item = [clothing_user_item, electronic_user_item, food_user_item]

user_bundle = [clothing_user_bundle, electronic_user_bundle, food_user_bundle]

session_bundle = [clothing_session_bundle, electronic_session_bundle, food_session_bundle]

session_item = [clothing_session_item, electronic_session_item, food_session_item]

item_names = [clothing_item_names, electronic_item_names, food_item_names]

In [14]:
with open("/content/LLM4BEAR/BundleRec Data/enriched_outputs_electronic.pkl", "rb") as f:
    text_electronics = pickle.load(f)

print(text_electronics[3498])

with open(f"/content/LLM4BEAR/BundleRec Data/enriched_outputs_clothing.pkl", "rb") as f:
    text_clothing = pickle.load(f)

print(text_clothing[3000])

with open(f"/content/LLM4BEAR/BundleRec Data/enriched_outputs_food.pkl", "rb") as f:
    text_food = pickle.load(f)

print(text_food[3000])


This device is a plug-and-play USB adapter that provides 5.1 channel surround sound capabilities to computers without the need for an internal sound card.
This is a pink swimsuit designed for girls aged 7 to 16, featuring a sweetheart neckline and suitable for swimming and beach activities.
A fragrant blend of star anise, cloves, Chinese cinnamon, Sichuan peppercorns, and ginger, this seasoning enhances a variety of dishes with its unique sweet and savory flavor profile.


In [ ]:
electronic_category_keys = [
    'Camera and Accessories', 'Computers and Accessories', 'Audio Equipment',
    'Tablets and Accessories', 'Storage Solutions', 'Networking Equipment',
    'Mobile Devices and Accessories', 'Travel Accessories', 'Gaming',
    'Home Entertainment Systems', 'Miscellaneous Electronics', 'Power Solutions',
    'Cables and Connectors', 'Security Systems', 'Car Technology and Accessories',
    'Photography and Camera Equipment', 'Adapters and Cables',
    'Television and Accessories', 'AV Setup', 'GPS and Navigation Accessories',
    'Mobile Device Protection', 'Streaming and Media', 'PC Building and Assembly',
    'General Electronics', 'Walkie Talkies and Communication Devices'
]


clothing_category_keys = [
    'Footwear', 'Accessories', 'Costumes and Themed Apparel',
    'Lingerie and Underwear', 'Baby and Kids Clothing', 'Activewear and Sportswear',
    'Fashion Accessories', 'Seasonal and Thematic Products', 'Electronics',
    'Children\'s Items', 'Carrying Items', 'Maintenance', 'Clothing'
]

food_category_keys = [
    'Snacks', 'Beverages', 'Cooking Ingredients',
    'Breakfast Foods', 'Sweets and Desserts', 'Health Foods',
    'Canned and Packaged Foods', 'Baby Food', 'Condiments and Sauces',
    'Fruits and Vegetables', 'Specialty Foods', 'Dried and Preserved Foods',
    'Grains and Pasta', 'Miscellaneous', 'Gift Baskets and Food Gifts',
    'Dietary Specific Items', 'Cooking Tools and Kitchen Goods',
    'Sweeteners', 'Nuts and Seeds', 'Coffee/Tea',
    'Vegetables and Beans', 'Health-Conscious Options', 'Prepared and Ready-Made Meals',
    'Ethnic and Specialty Foods', 'Culinary Specialties'
]


with open(f"/content/LLM4BEAR/BundleRec Data/specific_electronic_product_metadata.pkl", 'rb') as f:
    metadata_electronic_jsons = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/specific_clothing_product_metadata.pkl", 'rb') as f:
    metadata_clothing_jsons = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/specific_food_product_metadata.pkl", 'rb') as f:
    metadata_food_jsons = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/general_electronic_product_list.pkl", 'rb') as f:
    general_electronic_product_list = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/general_clothing_product_list.pkl", 'rb') as f:
    general_clothing_product_list = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/general_food_product_list.pkl", 'rb') as f:
    general_food_product_list = pickle.load(f)



In [12]:
def bundle_origin(bundle_ID, domain):

    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2

    original_session_ID = session_bundle[k].iloc[bundle_ID]["session ID"]

    item_ids = session_item[k][session_item[k]["session ID"] == original_session_ID]["item ID"].values


    # descript_sess = [metadata[k][metadata[k]['item ID'] == item_id].index.tolist()[0] for item_id in item_ids]

    descript_sess = [
    metadata[k][metadata[k]['item ID'] == item_id].index[0]
    for item_id in item_ids
    if (metadata[k]['item ID'] == item_id).any()
]


    return descript_sess



def bundle_str_generator(bundle_ID, domain, desc):


    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2

    bundle_list = bundle_items_list[k]

    descript_ids = [metadata[k][metadata[k]['titles'] == names].index.tolist()[0] for names in bundle_list[bundle_ID]]

    bundle_items =  [metadata[k].iloc[i]['titles'] for i in descript_ids]

    bundle_descriptions = [desc[i] for i in descript_ids]

    bundle_categories = [metadata[k].iloc[i]['categories'] for i in descript_ids]


    bundle_str = ""

    for i in range(len(bundle_items)):
        bundle_str = bundle_str + f"{i+1}. " + bundle_items[i] + ": " + bundle_descriptions[i] + "\n"

    return bundle_str, bundle_categories, bundle_items

def meta_str_generator(bundle_ID, domain, desc):


    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2

    bundle_list = bundle_items_list[k]

    descript_ids = [metadata[k][metadata[k]['titles'] == names].index.tolist()[0] for names in bundle_list[bundle_ID]]

    bundle_items =  [metadata[k].iloc[i]['titles'] for i in descript_ids]

    bundle_descriptions = [desc[i] for i in descript_ids]

    bundle_categories = [metadata[k].iloc[i]['categories'] for i in descript_ids]


    meta_list = [metadata_electronic_jsons[i] for i in descript_ids]

    token_list = []

    for i in range(len(descript_ids)):
        metasetter = json.loads(meta_list[i])

        key_features = metasetter['key_features']


        added_str = "[" + metasetter['product_type'] + "][" + metasetter['brand'] + "][" + metasetter['design_focus'] + "][" + metasetter['cost_tier'] + "]["# + metasetter['key_features']
        for j in range(len(key_features)):
            added_str = added_str + key_features[j] + "]["


        token_list.append(added_str[0:-1])



    bundle_str = ""

    for i in range(len(bundle_items)):
        bundle_str = bundle_str + f"{i+1}. " + bundle_items[i] + ": " + bundle_descriptions[i] + "\n"  + token_list[i] + "\n\n"

    return bundle_str, token_list

In [7]:
def item_token_generator(domain_length, metadata_list, domain):

    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2


    token_list = []

    for item_index in range(domain_length):

        if k == 0 or k == 1:

            meta_list = metadata_list[item_index]

            metasetter = json.loads(meta_list)

            key_features = metasetter['key_features']


            added_str = "[" + metasetter['product_type'] + "][" + metasetter['brand'] + "][" + metasetter['design_focus'] + "][" + metasetter['cost_tier'] + "]["# + metasetter['key_features']
            for j in range(len(key_features)):
                added_str = added_str + key_features[j] + "]["


            token_list.append(added_str[0:-1])

        elif k == 2:

            meta_list = metadata_list[item_index]

            metasetter = json.loads(meta_list)

            key_features = metasetter['key_features']

            dietary_features = metasetter['dietary_considerations']


            added_str = "[" + metasetter['product_type'] + "][" + metasetter['brand'] + "][" + metasetter['flavor_profile'] + "][" +  metasetter['cost_tier'] + "]["# + metasetter['key_features']

            for j in range(len(dietary_features)):
                added_str = added_str + dietary_features[j] + "]["

            for j in range(len(key_features)):
                added_str = added_str + key_features[j] + "]["


            token_list.append(added_str[0:-1])

    return token_list

# Decided to use the Specific Product Type Metadata.

In [ ]:
electronic_items = [electronic_metadata.iloc[i]['titles'] for i in range(len(text_electronics))]
all_electronic_item_tokens = item_token_generator(len(text_electronics), metadata_electronic_jsons, "electronic")

clothing_items = [clothing_metadata.iloc[i]['titles'] for i in range(len(text_clothing))]
all_clothing_item_tokens = item_token_generator(len(text_clothing), metadata_clothing_jsons, "clothing")

food_items = [food_metadata.iloc[i]['titles'] for i in range(len(text_food))]
all_food_item_tokens = item_token_generator(len(text_food), metadata_food_jsons, "food")

# with open(f"/content/drive/MyDrive/BundleRec Data/electronic_token_list.pkl", 'wb') as f:
#     pickle.dump(all_electronic_item_tokens, f)

# with open(f"/content/drive/MyDrive/BundleRec Data/clothing_token_list.pkl", 'wb') as f:
#     pickle.dump(all_clothing_item_tokens, f)

# with open(f"/content/drive/MyDrive/BundleRec Data/food_token_list.pkl", 'wb') as f:
#     pickle.dump(all_food_item_tokens, f)

In [ ]:
print(metadata_food_jsons[0])

In [4]:
with open(f"/content/LLM4BEAR/BundleRec Data/electronic_token_list.pkl", 'rb') as f:
    electronic_token_list = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/clothing_token_list.pkl", 'rb') as f:
    clothing_token_list = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/food_token_list.pkl", 'rb') as f:
    food_token_list = pickle.load(f)

In [5]:
for i in food_token_list[0:3]:
    print(i)

[hummus chips][Plocky's][savory][Medium-Cost][Gluten-Free][Vegan][single-serving][ready-to-eat][shelf-stable]
[fruit tea][Stash Tea][fruity][Medium-Cost][Caffeine-Free][foil-wrapped][ready-to-brew]
[stick pretzel][Rold Gold][savory][Low-Cost][Low-Fat][ready-to-eat][shelf-stable]


In [ ]:
from sentence_transformers import SentenceTransformer

# Load new top-tier model
model = SentenceTransformer("intfloat/e5-large-v2")
# OR
# modfrom sentence_transformers import SentenceTransformer



print("Model loaded!")


In [ ]:
# Generate embeddings
electronic_embeddings = model.encode(electronic_token_list, convert_to_tensor=False)  # Convert to NumPy array
# Save embeddings to file
np.save("/content/drive/MyDrive/BundleRec Data/electronic_item_token_embeddings.npy", electronic_embeddings)
print("Embeddings saved!")

# Generate embeddings
clothing_embeddings = model.encode(clothing_token_list, convert_to_tensor=False)  # Convert to NumPy array
# Save embeddings to file
np.save("/content/drive/MyDrive/BundleRec Data/clothing_item_token_embeddings.npy", clothing_embeddings)
print("Clothing embeddings saved!")


# Generate embeddings
food_embeddings = model.encode(food_token_list, convert_to_tensor=False)  # Convert to NumPy array
# Save embeddings to file
np.save("/content/drive/MyDrive/BundleRec Data/food_item_token_embeddings.npy", food_embeddings)
print("Food embeddings saved!")